# Discovery Challenges Clustering Notebook

This notebook loads cleaned challenge and expectation text, creates sentence embeddings, clusters challenge statements, and explores results with summary views and visualization.

In [3]:
%pip install -r ../requirements.txt
%pip install -q sentence-transformers
%pip install -q hdbscan
%pip install -q umap-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1) Environment Setup

Install or refresh the required packages for embeddings, clustering, and visualization.

## 2) Load Input Data

Read cleaned challenges and expectations, then remove rows missing challenge text.

In [4]:
import pandas as pd
import hdbscan

CHALLENGES_PATH = '../output/clean_challenges.csv'
EXPECTATIONS_PATH = '../output/clean_expectations.csv'

df = pd.read_csv(CHALLENGES_PATH)
expectations_df = pd.read_csv(EXPECTATIONS_PATH)

TEXT_COLUMN = "pain_points"

df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

## 3) Create Sentence Embeddings

Encode each pain point into a semantic vector using a transformer model.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")

challenge_embeddings = model.encode(
    df["pain_points"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = challenge_embeddings

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

## 4) Run HDBSCAN Clustering

Cluster embeddings to identify dense groups of similar challenge statements.

In [9]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric="euclidean"
)
df["Cluster"] = clusterer.fit_predict(embeddings)

## 5) Quick Cluster Diagnostics

Inspect noise points and basic cluster balance before deeper analysis.

In [ ]:
# THRESHOLD = 0.55

# assigned_themes = []
# theme_scores = []

# for row in similarity:

#     matches = []

#     for i, score in enumerate(row):

#         if score >= THRESHOLD:
#             matches.append(
#                 (
#                     theme_names[i],
#                     float(score)
#                 )
#             )

#     matches.sort(
#         key=lambda x: x[1],
#         reverse=True
#     )

#     assigned_themes.append(
#         [m[0] for m in matches[:3]]
#     )

#     theme_scores.append(
#         matches[:3]
#     )

# df["Themes"] = assigned_themes

# df["Theme Scores"] = theme_scores

In [10]:
noise = df[df.Cluster == -1]
print(f"Noise points: {len(noise)}")

Noise points: 92


## 6) Visualize Semantic Structure

Project embeddings to 2D for an interpretable cluster scatter plot.

In [11]:
from sklearn.decomposition import PCA
import plotly.express as px

reducer = PCA(n_components=2)

coords = reducer.fit_transform(embeddings)

plot_df = pd.DataFrame({
    "x": coords[:,0],
    "y": coords[:,1],
    "Cluster": df["Cluster"].astype(str),
    "Challenge": df["pain_points"]
})

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="Cluster",
    hover_data=["Challenge"],
    title="Semantic Clusters of Discovery Challenges"
)

fig.show(renderer="browser")

In [12]:
df["Cluster"].value_counts()

Cluster
 0    220
-1     92
 1     12
Name: count, dtype: int64

In [13]:
for cluster in sorted(df.Cluster.unique()):
    print("=" * 80)
    print(f"Cluster {cluster}")

    subset = df[df.Cluster == cluster]

    for text in subset["pain_points"].sample(min(15, len(subset)), random_state=42):
        print("•", text)

Cluster -1
• Inspector lists in the Periodic Inspection module are not alphabetized.
• Tablet Reliability Issues: Existing tablets are slow and unreliable.
• Scattered Information: Related property information is not presented in a single view.
• Difficult Certificate Research: Finding previous Certificates of Occupancy and capacity information requires extensive digging.
• No Automatic Cross-System Notifications: Permit and code workflows do not automatically notify each other.
• Navigation Difficulties: Users find Camino difficult to navigate.
• Outlook and IPS schedules are not synchronized.
• Document management and uploads can be inconsistent.
• No Automated Renewal Queue: Renewal management requires manual tracking.
• Complaint cases can take significant time to open.
• Remote Access Issues: Difficulties connecting to IPS remotely from the field.
• Limited Referral Tracking: IPS does not clearly identify where referrals originated or distinguish different referral types.
• Lead M

## 7) Similarity and Cluster Inspection

Review nearest-neighbor challenge similarity and print representative examples by cluster.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

challenge_similarity = cosine_similarity(challenge_embeddings)

In [ ]:
import numpy as np

for i in range(5):
    idx = np.argsort(challenge_similarity[i])[::-1][1:6]

    print("=" * 80)
    print(df.iloc[i]["pain_points"])
    print()

    print("Most similar:")

    for j in idx:
        print("-", df.iloc[j]["pain_points"])

In [ ]:
cluster_summary = (
    df.groupby("Cluster")
      .size()
      .reset_index(name="Count")
      .sort_values("Count", ascending=False)
)

cluster_summary

In [ ]:
for cluster in sorted(df.Cluster.unique()):

    if cluster == -1:
        continue

    print("="*80)
    print(f"Cluster {cluster}")

    subset = df[df.Cluster == cluster]

    print("Representative examples:")

    for text in subset["pain_points"].sample(
        min(8, len(subset)),
        random_state=42
    ):
        print("•", text)

In [ ]:
import umap
import plotly.express as px

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

coords = reducer.fit_transform(challenge_embeddings)

plot_df = pd.DataFrame({
    "x": coords[:,0],
    "y": coords[:,1],
    "Cluster": df["Cluster"].astype(str),
    "Challenge": df["pain_points"],
    "Focus Group": df["focus_group"]
})

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="Cluster",
    hover_data=["Focus Group","Challenge"],
    title="Semantic Clusters of Discovery Challenges"
)

fig.show()

## 8) Expectation Embeddings

Embed expectation statements for downstream alignment analysis against challenge clusters.

In [ ]:
expectation_embeddings = model.encode(
    expectations_df["expectations"].tolist(),
    normalize_embeddings=True
)